# video-notes: GPU OCR benchmark on Colab

Runs `frames.py` on the four hand-labelled test videos with three OCR modes and scores each:

- **tesseract**: the default (CPU)
- **hybrid**: Tesseract, plus RapidOCR on the talking-head review tiles to rescue overlays
- **rapidocr**: RapidOCR for every frame (GPU)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
Takes about 45–60 minutes. Keep the tab open.

## 1. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > T4 GPU")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("CPU cores:", __import__("os").cpu_count())

## 2. Install the tools and the skill

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -U yt-dlp rapidocr onnxruntime imagehash "opencv-python>=4.8,<5"
!rm -rf video-notes && git clone -q -b gpu-ocr https://github.com/endgame302/video-notes.git
!python video-notes/skills/video-notes/scripts/check_deps.py

## 3. Test videos

Downloads from YouTube. If YouTube blocks Colab ("Sign in to confirm you're not a bot"), put each
video in your Google Drive as `MyDrive/vn-bench/<id>/video.mp4` and run the next cell too.

In [ ]:
import json, pathlib, shutil, subprocess
IDS = ["YlY2g2xrl6Q", "BxV14h0kFs0", "jnaljahKibQ", "nEaJMdM1djM"]
WORK = pathlib.Path("/content/bench")
got = {}
for vid in IDS:
    f = WORK / vid / "video.mp4"
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        p = subprocess.run(["yt-dlp", "-q", "--no-warnings", "-f", "bv*[height<=1080][vcodec^=avc1]/bv*[height<=1080]",
                            "-o", str(f), f"https://www.youtube.com/watch?v={vid}"], capture_output=True, text=True)
        if p.returncode:
            print(vid, "download failed:", p.stderr.strip()[-300:])
    got[vid] = f.exists()
print("downloaded:", got)

In [ ]:
# Only if downloads failed: copy the videos from Google Drive
if not all(got.values()):
    from google.colab import drive
    drive.mount("/content/drive")
    for vid in IDS:
        src = pathlib.Path("/content/drive/MyDrive/vn-bench") / vid / "video.mp4"
        if not got[vid] and src.exists():
            shutil.copy(src, WORK / vid / "video.mp4")
            got[vid] = True
    print("available:", got)

## 4. Run the three OCR modes

In [ ]:
MODES = ["hybrid", "rapidocr", "tesseract"]
results = {}
for vid in [v for v in IDS if got[v]]:
    for mode in MODES:
        out = WORK / vid / f"frames_{mode}"
        p = subprocess.run(["python", "video-notes/skills/video-notes/scripts/frames.py", str(WORK / vid / "video.mp4"),
                            "--out", str(out), "--ocr", mode], capture_output=True, text=True)
        try:
            r = json.loads(p.stdout.strip().splitlines()[-1])
        except (IndexError, ValueError):
            r = {"error": (p.stderr or p.stdout)[-500:]}
        results[f"{vid}|{mode}"] = r
        print(vid, mode, r.get("secs"), r.get("device"), "rescued:", r.get("rescued"), r.get("error", ""), flush=True)

## 5. Score against the hand labels

In [ ]:
scores = {}
for mode in MODES:
    s = subprocess.run(["python", "video-notes/bench/score.py", str(WORK), "--manifest", f"frames_{mode}/frames.json"],
                       capture_output=True, text=True).stdout
    scores[mode] = s
    print("==", mode); print(s)

## 6. Summary (copy this back)

In [ ]:
summary = {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "cpus": __import__("os").cpu_count(), "runs": results, "scores": scores}
print("VN-BENCH-RESULT " + json.dumps(summary))